# Solving the Krusell-Smith (1998) Model in Julia

This notebook solves the Krusell-Smith (1998) model using value function iteration (VFI). The notebook is structured as follows:
1. Model setup (household, firm, equilibrium approximation).
2. Numerical algorithm description.
3. Julia implementation (VFI + simulation + law-of-motion update).

## 1) Original Model and Equilibrium

### HH problem

Individual state is $s=(k,e,z,\Phi)$, where $k$ is assets, $e\in\{0,1\}$ is employment, $z\in\{z_b,z_g\}$ is aggregate productivity, and $\Phi$ is the distribution of wealth and employment across agents.

Given prices $r(z,\Phi)$ and $w(z,\Phi)$, and law of motion for the distribution $\Phi'$, the Bellman equation is
$$
V(k,e,z,\Phi)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',\Phi')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,\Phi))k + w(z,\Phi)e \\
k'\ge k_{min} \\
\Phi' = \mathcal{H}(\Phi,z,z')
$$

However, the distribution $\Phi$ is infinite-dimensional, so we cannot solve this model numerically. 

Instead, we follow Krusell-Smith and approximate the distribution with a finite set of moments, which here we denote $K$ (aggregate capital). 

## 2) Model with Krusell-Smith Approximation

### HH problem
Individual state is $s=(k,e,z,K)$, where $k$ is assets, $e\in\{0,1\}$ is employment, $z\in\{z_b,z_g\}$ is aggregate productivity, and $K$ is aggregate capital.
Given prices and an aggregate forecasting rule, the Bellman equation is
$$
V(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',K')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,K,L))k + w(z,K,L)e \\
k'\ge k_{min} \\
K' = \mathcal{H}(K,z)
$$


### Firms problem
We assume a representative firm with production function:
$$
Y = zK^\alpha N^{1-\alpha},
$$
and competitive factor prices are
$$
r = \alpha z (K/N)^{\alpha-1}-\delta,\qquad
w = (1-\alpha)z(K/N)^\alpha.
$$

### Equilibrium definition
A recursive equilibrium is a collection
$$
\{V,\,g,\mathcal{H},\,r,\,w,\,\Phi\},
$$

such that:
1. Given $(r(z,K,L),w(z,K,L))$ and perceived law of motion $\mathcal{H}$, value function $V$ and policy function $g$ solve the household problem.
2. Given $z$ and $r(z,K,L), w(z,K,L)$, firm maximizes profits, in particular:
$$
r(z,K,L) = \alpha z (K/N)^{\alpha-1}-\delta,\qquad
w(z,K,L) = (1-\alpha)z(K/N)^\alpha.
$$
3. Given $g$ and shock transitions, we can obtain the actual law of motion for aggregate capital:
    $$
    K' = \mathcal{G}(K,z)
    $$
    The actual law of motion $\mathcal{G}$ is consistent with the perceived law of motion $\mathcal{H}$
4. Market clearing:
$$
K = \int k\,d\Phi(k,e,z,K) \\
N = \int e\,d\Phi(k,e,z,K)
$$

Krusell-Smith approximation specifies a particular functional form for the perceived law of motion $\mathcal{H}$, which is log-linear:
$$
\log K' = a_z + b_z \log K
$$

## 3) General Numerical Algorithm (Inner/Outer Iteration)

We use a nested iteration algorithm.

### Inner loop: solve HH problem
Given coefficients $(a_z,b_z)$, we map $K$ to predicted $K'$ and solve HH problem:
$$
V(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',K')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,K,L))k + w(z,K,L)e \\
k'\ge k_{min} \\
\log K' = a_z + b_z \log K
$$

At each state $(k,e,z,K)$ we search over $k'\in\mathcal K$ and compute expected continuation value using transition probabilities for $(z,e)\to(z',e')$.
We iterate until value function converged:
$$
\|V_{new}-V\|_\infty < \varepsilon_V.
$$

### Outer loop: update aggregate law of motion
With the policy function $g(k,e,z,K)$ solved from the inner loop, we:
1. Simulate many households and periods.
2. Construct time series data $\{K_t,z_t\}$.
3. Run state-by-state regressions
$$
\log K_{t+1}=a_z+b_z\log K_t+\epsilon_{t+1},\qquad z\in\{b,g\}.
$$
4. Update perceived coefficients with damping
$$
\theta^{new}=\lambda\theta^{old}+(1-\lambda)\hat\theta,\quad \theta=(a_b,b_b,a_g,b_g).
$$
5. Repeat until stopping criterion is met (in this notebook: both $R^2$ exceed a threshold).


---

## Implementation in Julia

## 1) Calibration

### 1) Structural parameters
We use:
- Discount factor: $\beta = 0.99$
- Risk aversion: $\sigma = 1$ (log utility)
- Capital share: $\alpha = 0.36$
- Depreciation: $\delta = 0.025$
- Ad hoc borrowing constraint: $k_{min} = 0$
- Aggregate shock values: $z_b=0.99$, $z_g=1.01$
- Hours of labor supply when employed: $\tilde{l}= 0.3271$

We have these following calibration targets:
- Unemployment rates: $u_b=0.10$, $u_g=0.04$
- Average duration of both good and bad times is eight quarters
- Average duration of unemployment is 1.5 quarters in good times and 2.5 quarters in bad times.

### 2) Aggregate productivity shocks
- Aggregate productivity is two-state:
    $$
    z_t \in \{z_b, z_g\}, \quad z_b=0.99,\; z_g=1.01.
    $$

    The transition matrix is:
    $$
    \Pi_z = \begin{bmatrix} \pi_{bb} & \pi_{bg} \\
    \pi_{gb} & \pi_{gg} \end{bmatrix}
    $$


- Aggregate unemployment rates:
  
    We assume different unemployment rates in the two aggregate states:
    $$
    u_b=0.10,\quad u_g=0.04.
    $$
    In the bad state, 10% of agents are unemployed, while in the good state only 4% are unemployed.

- Effective aggregate labor in each state:
    $$
    L_b=\tilde{l}(1-u_b) = 0.2944, \\
    L_g=\tilde{l}(1-u_g) = 0.3142.
    $$

### 3) Calibrating the aggregate transition matrix.

- First, calibrate the transition matrix for the aggregate shock $z$.

    Average duration of each aggregate state is 8 quarters, we know that:

    The duration of state $s$ can be computed:
    $$
    \text{Duration}_s = \sum_{t=1}^\infty t \cdot (1-\pi_{ss}) \pi_{ss}^{t-1} = \frac{1}{1-\pi_{ss}}
    $$
    Thus
    $$
    \pi_{bb} = \pi_{gg} = 1 - \frac{1}{8} = 0.875, \qquad
    \pi_{bg} = \pi_{gb} = 0.125.
    $$

### 4) Calibrating the joint transition matrix for $(z,e)$.

Then, we calibrate the joint transition matrix for $(z,e)$
  
We assume the individual shock $e$ is correlated with the aggregate shock $z$, thus we need to specify the joint transition matrix for $(z,e)$.
    
Addtionally, we assume that given $z$, transition of individual employment status $e$ is independent cross agents:

Thus, we have $\forall s,s'\in\{b,g\}$:
$$
\pi_{ss'00} +  \pi_{ss'01} = \pi_{ss'10} + \pi_{ss'11} = \pi_{ss'}
$$
and
$$
u_s \frac{\pi_{ss'00}}{\pi_{ss'}} + (1-u_s)\frac{\pi_{ss'10}}{\pi_{ss'}} = u_{s'}
$$

From transition matrix of $z$, we have:
$$
\pi_{bb00} + \pi_{bb01} = 0.875, \\
\pi_{bb10} + \pi_{bb11} = 0.875, \\
\pi_{bg00} + \pi_{bg01} = 0.125, \\
\pi_{bg10} + \pi_{bg11} = 0.125, \\
\pi_{gb00} + \pi_{gb01} = 0.125, \\
\pi_{gb10} + \pi_{gb11} = 0.125, \\
\pi_{gg00} + \pi_{gg01} = 0.875, \\
\pi_{gg10} + \pi_{gg11} = 0.875.
$$

We already know that average unemployment duration is 1.5 quarters in good times and 2.5 quarters in bad times, thus we have:
$$
\pi_{bb00} = 1 - \frac{1}{2.5} = 0.6, \\
\pi_{gg00} = 1 - \frac{1}{1.5} = 1/3.
$$
Implies
$$
\pi_{bb01} = 0.875 - \pi_{bb00} = 0.275, \\
\pi_{gg01} = 0.875 - \pi_{gg00} = 0.542.
$$

Then we impose additional restrictions to solve for the rest of the transition probabilities:
$$
\frac{\pi_{gb00}}{\pi_{bb00}} = 1.25 \frac{\pi_{gb}}{\pi_{bb}} \Rightarrow \\
\pi_{gb00} = 1.25 \cdot \frac{0.125}{0.875} \cdot 0.6 = 0.107, \\
\pi_{gb01} = 0.125 - \pi_{gb00} = 0.018,
$$

Another additional restriction:
$$
\frac{\pi_{bg00}}{\pi_{gg00}} = 0.75 \frac{\pi_{bg}}{\pi_{gg}} \Rightarrow \quad \\
\pi_{bg00} = 0.75 \cdot \frac{0.125}{0.875} \cdot \frac{1}{3} = 0.036, \\
\pi_{bg01} = 0.125 - \pi_{bg00} = 0.089, \\
$$

Remaining probabilities have to be such that unemployment is always constant:
$$
u_s \frac{\pi_{ss'00}}{\pi_{ss'}} + (1-u_s)\frac{\pi_{ss'10}}{\pi_{ss'}} = u_{s'}
$$

Thus, we have:
$$
u_g \frac{\pi_{gg00}}{\pi_{gg}} + (1-u_g)\frac{\pi_{gg10}}{\pi_{gg}} = u_g \\
u_g \frac{\pi_{gb00}}{\pi_{gb}} + (1-u_g)\frac{\pi_{gb10}}{\pi_{gb}} = u_b \\
u_b \frac{\pi_{bb00}}{\pi_{bb}} + (1-u_b)\frac{\pi_{bb10}}{\pi_{bb}} = u_b \\
u_b \frac{\pi_{bg00}}{\pi_{bg}} + (1-u_b)\frac{\pi_{bg10}}{\pi_{bg}} = u_g
$$

Plug in parameters we know and solve for the rest of the transition probabilities:
$$
0.04 \cdot \frac{1/3}{0.875} + 0.96 \cdot \frac{\pi_{gg10}}{0.875} = 0.04 \quad \Rightarrow \quad \pi_{gg10} = 0.022 \quad \Rightarrow \quad \pi_{gg11} = 0.875 - 0.022 = 0.853 \\
0.04 \cdot \frac{0.107}{0.125} + 0.96 \cdot \frac{\pi_{gb10}}{0.125} = 0.10 \quad \Rightarrow \quad \pi_{gb10} = 0.008 \quad \Rightarrow \quad \pi_{gb11} = 0.125 - 0.008 = 0.117 \\
0.10 \cdot \frac{0.6}{0.875} + 0.90 \cdot \frac{\pi_{bb10}}{0.875} = 0.10 \quad \Rightarrow \quad \pi_{bb10} = 0.030 \quad \Rightarrow \quad \pi_{bb11} = 0.875 - 0.030 = 0.845 \\
0.10 \cdot \frac{0.036}{0.125} + 0.90 \cdot \frac{\pi_{bg10}}{0.125} = 0.04 \quad \Rightarrow \quad \pi_{bg10} = 0.002 \quad \Rightarrow \quad \pi_{bg11} = 0.125 - 0.002 = 0.123
$$

Thus, the joint transition matrix for $(z,e)$ is:
$$
P=
\begin{bmatrix}
\pi_{bb00} & \pi_{bb01} & \pi_{bg00} & \pi_{bg01}\\
\pi_{bb10} & \pi_{bb11} & \pi_{bg10} & \pi_{bg11}\\
\pi_{gb00} & \pi_{gb01} & \pi_{gg00} & \pi_{gg01}\\
\pi_{gb10} & \pi_{gb11} & \pi_{gg10} & \pi_{gg11}
\end{bmatrix}.
$$

Substituting in the calibrated values gives

$$
P \approx
\begin{bmatrix}
0.600 & 0.275 & 0.036 & 0.089\\
0.031 & 0.844 & 0.002 & 0.123\\
0.107 & 0.018 & 0.333 & 0.542\\
0.009 & 0.116 & 0.023 & 0.852
\end{bmatrix}.
$$

